首先，我们将了解 PyTorch Geometric 如何将图存储为 PyTorch 张量。

然后，我们将使用 ogb 包加载和检查其中一个 Open Graph Benchmark (OGB) 数据集。OGB 是用于图机器学习的现实、大规模和多样化的基准数据集的集合。ogb 包不仅为每个数据集提供数据加载器，还提供模型评估器。

最后，我们将使用 PyTorch Geometric 构建我们自己的 GNN。然后，我们将在 OGB 节点属性预测和图形属性预测任务上训练和评估我们的模型。

注意：确保按顺序运行每个部分中的所有单元，以便中间变量/包将延续到下一个单元 完成本次实验的时间约为两小时

# 环境搭建

In [6]:
import os

import torch
print("PyTorch has version {}".format(torch.__version__))

PyTorch has version 2.5.1+cu121


下载 PyG 的依赖，确保其与 torch 下载的版本契合，如果有问题可以查阅文档 [PyG's page](https://www.google.com/url?q=https%3A%2F%2Fpytorch-geometric.readthedocs.io%2Fen%2Flatest%2Fnotes%2Finstallation.html)

In [7]:
# 安装 torch geometric
import os
import torch
if 'IS_GRADESCOPE_ENV' not in os.environ:
  torch_version = str(torch.__version__)
  scatter_src = f"https://pytorch-geometric.com/whl/torch-{torch_version}.html"
  sparse_src = f"https://pytorch-geometric.com/whl/torch-{torch_version}.html"
  !pip install torch-scatter -f $scatter_src
  !pip install torch-sparse -f $sparse_src
  !pip install torch-geometric
  !pip install ogb

Looking in links: https://pytorch-geometric.com/whl/torch-2.5.1+cu121.html
Looking in links: https://pytorch-geometric.com/whl/torch-2.5.1+cu121.html


# 1)PyG (数据集和数据)

PyTorch Geometric 有两个用于存储和/或将图转换为张量格式的类。
一个是 `torch_geometric.datasets`，它包含了各种常见的图数据集；
另一个是 `torch_geometric.data`，它提供了将图转换为 PyTorch 张量的相关数据处理功能。

在本节中，我们将学习如何将 `torch_geometric.datasets` 和 `torch_geometric.data` 结合使用。

## PyG 数据集

`torch_geometric.datasets` 类有许多图数据集，我们使用其一来探索其用法

In [8]:
from torch_geometric.datasets import TUDataset

if 'IS_GRADESCOPE_ENV' not in os.environ:
  root = './enzymes'
  name = 'ENZYMES'

  # ENZYMES(酶)数据集
  pyg_dataset= TUDataset(root, name)

  # 其中有六百个图
  print(pyg_dataset)

ENZYMES(600)


### Question1: ENZYMES 数据集中有多少类，多少特征

In [10]:
def get_num_classes(pyg_dataset):
  num_classes = pyg_dataset.num_classes
  return num_classes

def get_num_features(pyg_dataset):
  num_features = pyg_dataset.num_features
  return num_features

if 'IS_GRADESCOPE_ENV' not in os.environ:
  num_classes = get_num_classes(pyg_dataset)
  num_features = get_num_features(pyg_dataset)
  print("{} dataset has {} classes".format(name, num_classes))
  print("{} dataset has {} features".format(name, num_features))


ENZYMES dataset has 6 classes
ENZYMES dataset has 3 features


## PyG 数据

每个 PyG 数据集都存储了一个由 `torch_geometric.data.Data` 对象组成的列表，其中每个 `torch_geometric.data.Data` 对象表示一张图。

我们可以通过索引数据集获取 `Data` 对象。 
关于 `Data` 对象中包含哪些信息等更多内容，请参考[官方文档](https://pytorch-geometric.readthedocs.io/en/latest/modules/data.html#torch_geometric.data.Data)。

### Question 2： ENZYMES 数据集中 index 为 100 的图的 label 是什么？

In [11]:
def get_graph_class(pyg_dataset, idx):
  label = -1
  label = pyg_dataset[idx].y.item()
  return label

# 此处的 pyg_dataset 是用于图分类的数据集
if 'IS_GRADESCOPE_ENV' not in os.environ:
  graph_0 = pyg_dataset[0]
  print(graph_0)
  idx = 100
  label = get_graph_class(pyg_dataset, idx)
  print('Graph with index {} has label {}'.format(idx, label))

Data(edge_index=[2, 168], x=[37, 3], y=[1])
Graph with index 100 has label 4


### Question 3：index 为 200 的图有多少条边？

In [12]:
def get_graph_num_edges(pyg_dataset, idx):

  num_edges = 0
  
  graph = pyg_dataset[idx]
  edge_index = graph.edge_index
  # 由于是无向图，每条边在edge_index中出现两次
  num_edges = edge_index.shape[1] // 2

  return num_edges

if 'IS_GRADESCOPE_ENV' not in os.environ:
  idx = 200
  num_edges = get_graph_num_edges(pyg_dataset, idx)
  print('Graph with index {} has {} edges'.format(idx, num_edges))

Graph with index 200 has 53 edges


# 2) Open Graph Benchmark(OGB)

**Open Graph Benchmark（OGB）** 是一个用于图机器学习的现实、大规模且多样化的基准数据集集合。

这些数据集可以通过 OGB 的数据加载器（OGB Data Loader）**自动下载、处理并划分**。

随后，可以使用 OGB 的评估器（OGB Evaluator）以统一的方式对模型性能进行评估。

如果数据集自动下载速度较慢，可以从Nju Box下载：https://box.nju.edu.cn/d/5f1c0015382643c9be0d/

## 数据集和数据

OGB 也支持 PyG 的数据集/数据的类。此处我们查看 `ogbn-arxiv` 数据集

In [13]:
import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset



if 'IS_GRADESCOPE_ENV' not in os.environ:
  dataset_name = 'ogbn-arxiv'
  # 加载数据集并转换为稀疏图
  dataset = PygNodePropPredDataset(name=dataset_name,
                                  transform=T.ToSparseTensor())
  print('The {} dataset has {} graph'.format(dataset_name, len(dataset)))

  # 分离一张图出来
  data = dataset[0]
  print(data)

Downloaded 0.08 GB: 100%|██████████| 81/81 [00:06<00:00, 12.20it/s]


Extracting dataset\arxiv.zip


Processing...


Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 1/1 [00:00<?, ?it/s]


Converting graphs into PyG objects...


100%|██████████| 1/1 [00:00<00:00, 272.78it/s]

Saving...
The ogbn-arxiv dataset has 1 graph



Done!
c:\Users\roy-h\.conda\envs\pytorch\lib\site-packages\ogb\nodeproppred\dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.

Data(num_nodes=169343, x=[169343, 128], node_year=[169343, 1], y=[169343, 1], adj_t=[169343, 169343, nnz=1166243])


### Question 4: ogbn-arxiv 的图中有多少特征？

In [14]:
def graph_num_features(data):
  num_features = 0
  num_features = data.x.shape[1]

  return num_features

if 'IS_GRADESCOPE_ENV' not in os.environ:
  num_features = graph_num_features(data)
  print('The graph has {} features'.format(num_features))

The graph has 128 features


# 3） GNN：节点属性预测

在本节中，我们将使用 PyTorch Geometric 构建第一个图神经网络。然后，我们会将其应用于**节点属性预测（节点分类）**任务。

具体来说，我们将以 **GCN（图卷积网络）** 作为图神经网络的基础（参考 [Kipf 等人, 2017](https://arxiv.org/abs/1609.02907)）。  
为此，我们将使用 PyG 内置的 `GCNConv` 层。

## 环境搭建

In [15]:
import torch
import pandas as pd
import torch.nn.functional as F
print(torch.__version__)

# 使用 PyG 内建的 GCNConv
from torch_geometric.nn import GCNConv

import torch_geometric.transforms as T
from ogb.nodeproppred import PygNodePropPredDataset, Evaluator

2.5.1+cu121


## 加载并处理数据

In [16]:
if 'IS_GRADESCOPE_ENV' not in os.environ:
  dataset_name = 'ogbn-arxiv'
  dataset = PygNodePropPredDataset(name=dataset_name,
                                  transform=T.Compose([T.ToUndirected(),T.ToSparseTensor()]))
  data = dataset[0]

  device = 'cuda' if torch.cuda.is_available() else 'cpu'

  # 如果你在使用 gpu ， device 应该是 cuda
  print('Device: {}'.format(device))

  data = data.to(device)
  split_idx = dataset.get_idx_split()
  train_idx = split_idx['train'].to(device)

c:\Users\roy-h\.conda\envs\pytorch\lib\site-packages\ogb\nodeproppred\dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.load(se

Device: cuda


## GCN 模型

现在我们来实现我们的 GCN 模型！

请根据下图所示的结构来实现 `forward` 函数：
![GCN 模型结构图](https://drive.google.com/uc?id=128AuYAXNXGg7PIhJJ7e420DoPWKb-RtL)

In [17]:
class GCN(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers,
                 dropout, return_embeds=False):
        super(GCN, self).__init__()

        # 一个包含 GCNConv 层的列表
        self.convs = torch.nn.ModuleList()
        
        # 添加第一层 (输入层到隐藏层)
        self.convs.append(GCNConv(input_dim, hidden_dim))
        
        # 添加中间层 (隐藏层到隐藏层)
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_dim, hidden_dim))
            
        # 添加最后一层 (隐藏层到输出层)
        self.convs.append(GCNConv(hidden_dim, output_dim))

        # 一个包含一维批归一化层（BatchNorm1d）的列表
        self.bns = torch.nn.ModuleList()
        for _ in range(num_layers - 1):
            self.bns.append(torch.nn.BatchNorm1d(hidden_dim))

        # log softmax 层
        self.softmax = torch.nn.LogSoftmax(dim=1)

        # 元素被置为 0 的概率（Dropout 概率）
        self.dropout = dropout

        # 是否跳过分类层并返回节点嵌入
        self.return_embeds = return_embeds

    def reset_parameters(self):
        for conv in self.convs:
            conv.reset_parameters()
        for bn in self.bns:
            bn.reset_parameters()

    def forward(self, x, adj_t):
        out = x
        
        # 处理前 n-1 层，每层后接BN、ReLU和Dropout
        for i in range(len(self.convs) - 1):
            out = self.convs[i](out, adj_t)
            out = self.bns[i](out)
            out = F.relu(out)
            out = F.dropout(out, p=self.dropout, training=self.training)
        
        # 最后一层
        out = self.convs[-1](out, adj_t)
        
        # 如果不返回嵌入，则应用softmax
        if not self.return_embeds:
            out = self.softmax(out)

        return out

In [18]:
def train(model, data, train_idx, optimizer, loss_fn):
    model.train()
    loss = 0

    # 优化器梯度清零
    optimizer.zero_grad()
    # 将数据输入模型
    out = model(data.x, data.adj_t)
    # 使用train_idx对模型输出和标签进行切片
    loss = loss_fn(out[train_idx], data.y.squeeze(1)[train_idx])
    
    loss.backward()
    optimizer.step()

    return loss.item()

In [19]:
@torch.no_grad()
def test(model, data, split_idx, evaluator, save_model_results=False):
    model.eval()

    # 模型在所有数据上的输出
    out = model(data.x, data.adj_t)

    y_pred = out.argmax(dim=-1, keepdim=True)

    train_acc = evaluator.eval({
        'y_true': data.y[split_idx['train']],
        'y_pred': y_pred[split_idx['train']],
    })['acc']
    valid_acc = evaluator.eval({
        'y_true': data.y[split_idx['valid']],
        'y_pred': y_pred[split_idx['valid']],
    })['acc']
    test_acc = evaluator.eval({
        'y_true': data.y[split_idx['test']],
        'y_pred': y_pred[split_idx['test']],
    })['acc']

    if save_model_results:
      print ("Saving Model Predictions")

      data = {}
      data['y_pred'] = y_pred.view(-1).cpu().detach().numpy()

      df = pd.DataFrame(data=data)
      # 本地保存为 CSV 文件
      df.to_csv('ogbn-arxiv_node.csv', sep=',', index=False)

    return train_acc, valid_acc, test_acc

In [20]:
# 请不要改变 args
if 'IS_GRADESCOPE_ENV' not in os.environ:
  args = {
      'device': device,
      'num_layers': 3,
      'hidden_dim': 256,
      'dropout': 0.5,
      'lr': 0.01,
      'epochs': 100,
  }
  args

In [21]:
if 'IS_GRADESCOPE_ENV' not in os.environ:
  model = GCN(data.num_features, args['hidden_dim'],
              dataset.num_classes, args['num_layers'],
              args['dropout']).to(device)
  evaluator = Evaluator(name='ogbn-arxiv')

In [22]:
# 请不要改变 args
# 使用 GPU 训练应该小于 10 分钟
import copy
if 'IS_GRADESCOPE_ENV' not in os.environ:
  # reset the parameters to initial random value
  model.reset_parameters()

  optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'])
  loss_fn = F.nll_loss

  best_model = None
  best_valid_acc = 0

  for epoch in range(1, 1 + args["epochs"]):
    loss = train(model, data, train_idx, optimizer, loss_fn)
    result = test(model, data, split_idx, evaluator)
    train_acc, valid_acc, test_acc = result
    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        best_model = copy.deepcopy(model)
    print(f'Epoch: {epoch:02d}, '
          f'Loss: {loss:.4f}, '
          f'Train: {100 * train_acc:.2f}%, '
          f'Valid: {100 * valid_acc:.2f}% '
          f'Test: {100 * test_acc:.2f}%')

Epoch: 01, Loss: 3.9760, Train: 24.98%, Valid: 28.41% Test: 25.58%
Epoch: 02, Loss: 2.3340, Train: 23.97%, Valid: 20.91% Test: 25.71%
Epoch: 03, Loss: 1.9186, Train: 19.53%, Valid: 10.03% Test: 8.25%
Epoch: 04, Loss: 1.7623, Train: 20.72%, Valid: 10.87% Test: 8.78%
Epoch: 05, Loss: 1.6325, Train: 26.63%, Valid: 17.31% Test: 14.76%
Epoch: 06, Loss: 1.5582, Train: 34.49%, Valid: 25.84% Test: 23.74%
Epoch: 07, Loss: 1.5042, Train: 36.37%, Valid: 22.03% Test: 19.01%
Epoch: 08, Loss: 1.4411, Train: 38.32%, Valid: 23.52% Test: 20.14%
Epoch: 09, Loss: 1.3948, Train: 41.40%, Valid: 29.54% Test: 27.41%
Epoch: 10, Loss: 1.3615, Train: 44.80%, Valid: 38.42% Test: 39.56%
Epoch: 11, Loss: 1.3306, Train: 46.18%, Valid: 42.02% Test: 45.84%
Epoch: 12, Loss: 1.3034, Train: 46.30%, Valid: 42.91% Test: 47.34%
Epoch: 13, Loss: 1.2812, Train: 47.30%, Valid: 45.52% Test: 49.90%
Epoch: 14, Loss: 1.2644, Train: 49.81%, Valid: 50.32% Test: 53.98%
Epoch: 15, Loss: 1.2452, Train: 52.06%, Valid: 54.04% Test: 56.5

### Question 5 ：你的**最佳模型**验证集和测试集精度如何？

运行下面的代码单元格，可以查看你最优模型的预测结果，  
并将模型的预测保存到名为 `ogbn-arxiv_node.csv` 的文件中。

In [23]:
if 'IS_GRADESCOPE_ENV' not in os.environ:
  best_result = test(best_model, data, split_idx, evaluator, save_model_results=True)
  train_acc, valid_acc, test_acc = best_result
  print(f'Best model: '
        f'Train: {100 * train_acc:.2f}%, '
        f'Valid: {100 * valid_acc:.2f}% '
        f'Test: {100 * test_acc:.2f}%')

Saving Model Predictions
Best model: Train: 73.74%, Valid: 71.91% Test: 71.15%


# 4） GNN：图性质预测

在这一节中我们将创建一个为图性质预测的 GNN

## 加载并预处理数据集

In [24]:
from ogb.graphproppred import PygGraphPropPredDataset, Evaluator
from torch_geometric.data import DataLoader
from tqdm import tqdm

if 'IS_GRADESCOPE_ENV' not in os.environ:
  # 加载数据集
  dataset = PygGraphPropPredDataset(name='ogbg-molhiv')

  device = 'cuda' if torch.cuda.is_available() else 'cpu'
  print('Device: {}'.format(device))

  split_idx = dataset.get_idx_split()

  # 检查任务类型
  print('Task type: {}'.format(dataset.task_type))

Downloaded 0.00 GB: 100%|██████████| 3/3 [00:02<00:00,  1.49it/s]
Processing...


Extracting dataset\hiv.zip
Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 41127/41127 [00:00<00:00, 174770.46it/s]


Converting graphs into PyG objects...


100%|██████████| 41127/41127 [00:00<00:00, 50108.74it/s]


Saving...
Device: cuda
Task type: binary classification


Done!
c:\Users\roy-h\.conda\envs\pytorch\lib\site-packages\ogb\graphproppred\dataset_pyg.py:68: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.

In [25]:
# 将数据集划分加载到对应的 dataloader 中
# 我们将在每批 32 个图上进行图分类任务的训练
# 对训练集中的图顺序进行打乱
if 'IS_GRADESCOPE_ENV' not in os.environ:
  train_loader = DataLoader(dataset[split_idx["train"]], batch_size=32, shuffle=True, num_workers=0)
  valid_loader = DataLoader(dataset[split_idx["valid"]], batch_size=32, shuffle=False, num_workers=0)
  test_loader = DataLoader(dataset[split_idx["test"]], batch_size=32, shuffle=False, num_workers=0)

c:\Users\roy-h\.conda\envs\pytorch\lib\site-packages\torch_geometric\deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


In [26]:
if 'IS_GRADESCOPE_ENV' not in os.environ:
  # Please do not change the args
  args = {
      'device': device,
      'num_layers': 5,
      'hidden_dim': 256,
      'dropout': 0.5,
      'lr': 0.001,
      'epochs': 30,
  }
  args

## 图预测模型

图的 Mini-Batching（小批量处理）

在正式进入模型之前，我们先介绍图数据的 mini-batching 概念。为了并行处理一小批图，  
PyG 会将这些图组合成一个**不相连的大图**数据对象（`torch_geometric.data.Batch`）。

`torch_geometric.data.Batch` 继承自之前介绍的 `torch_geometric.data.Data`，  
并额外包含一个名为 `batch` 的属性。

这个 `batch` 属性是一个向量，用来将每个节点映射到它在 mini-batch 中所属图的索引。例如：
<code>batch = [0, ..., 0, 1, ..., 1, ..., n - 2, n - 1, ..., n - 1]<code>

这个属性非常重要，它能帮助我们知道每个节点属于哪个图。  
举个例子，它可以用来对每个图的节点嵌入进行平均，从而得到图级别的嵌入表示。

### 补全

现在，我们已经具备了实现 GCN 图预测模型所需的所有工具！

我们将复用现有的 GCN 模型来生成 **节点嵌入（node_embeddings）**，  
然后对节点进行 **全局池化（Global Pooling）**，从而得到每个图的**图级别嵌入（graph level embeddings）**，  
这些嵌入将用于预测每个图的属性。

请记住，`batch` 属性对于在 mini-batch 中执行全局池化操作是很重要的。

In [32]:
from ogb.graphproppred.mol_encoder import AtomEncoder
from torch_geometric.nn import global_add_pool, global_mean_pool

### GCN 用于预测图属性
class GCN_Graph(torch.nn.Module):
    def __init__(self, hidden_dim, output_dim, num_layers, dropout):
        super(GCN_Graph, self).__init__()

        # 加载分子图中原子的编码器
        self.node_encoder = AtomEncoder(hidden_dim)

        # 节点嵌入模型
        # 注意：输入维度和输出维度都设置为 hidden_dim
        self.gnn_node = GCN(hidden_dim, hidden_dim,
            hidden_dim, num_layers, dropout, return_embeds=True)

        # 将 self.pool 直接初始化为全局平均池化层
        self.pool = global_mean_pool

        # 输出层
        self.linear = torch.nn.Linear(hidden_dim, output_dim)


    def reset_parameters(self):
        self.gnn_node.reset_parameters()
        self.linear.reset_parameters()

    def forward(self, batched_data):
        # 获取节点特征和边索引
        x, edge_index, batch = batched_data.x, batched_data.edge_index, batched_data.batch
        
        # 对节点特征进行编码
        x = self.node_encoder(x)
        
        # 通过GNN获取节点嵌入
        node_embeddings = self.gnn_node(x, edge_index)
        
        # 使用池化操作获取图级别嵌入
        graph_embeddings = self.pool(node_embeddings, batch)
        
        # 通过线性层得到最终预测
        pred = self.linear(graph_embeddings)
        
        return pred

In [33]:
def train(model, device, data_loader, optimizer, loss_fn):
    model.train()
    loss = 0

    for step, batch in enumerate(tqdm(data_loader, desc="Iteration")):
      batch = batch.to(device)

      if batch.x.shape[0] == 1 or batch.batch[-1] == 0:
          pass
      else:
        is_labeled = batch.y == batch.y
        
        optimizer.zero_grad()
        pred = model(batch)
        loss = loss_fn(pred[is_labeled], batch.y[is_labeled].to(torch.float32))

        loss.backward()
        optimizer.step()

    return loss.item()

In [34]:
# 用于分析的函数
def eval(model, device, loader, evaluator, save_model_results=False, save_file=None):
    model.eval()
    y_true = []
    y_pred = []

    for step, batch in enumerate(tqdm(loader, desc="Iteration")):
        batch = batch.to(device)

        if batch.x.shape[0] == 1:
            pass
        else:
            with torch.no_grad():
                pred = model(batch)

            y_true.append(batch.y.view(pred.shape).detach().cpu())
            y_pred.append(pred.detach().cpu())

    y_true = torch.cat(y_true, dim = 0).numpy()
    y_pred = torch.cat(y_pred, dim = 0).numpy()

    input_dict = {"y_true": y_true, "y_pred": y_pred}

    if save_model_results:
        print ("Saving Model Predictions")

        # 创建一个包含两列的 pandas 数据框（DataFrame）
        # y_pred | y_true
        data = {}
        data['y_pred'] = y_pred.reshape(-1)
        data['y_true'] = y_true.reshape(-1)

        df = pd.DataFrame(data=data)
        # Save to csv
        df.to_csv('ogbg-molhiv_graph_' + save_file + '.csv', sep=',', index=False)

    return evaluator.eval(input_dict)

In [35]:
if 'IS_GRADESCOPE_ENV' not in os.environ:
  model = GCN_Graph(args['hidden_dim'],
              dataset.num_tasks, args['num_layers'],
              args['dropout']).to(device)
  evaluator = Evaluator(name='ogbg-molhiv')

In [36]:
import copy

if 'IS_GRADESCOPE_ENV' not in os.environ:
  model.reset_parameters()

  optimizer = torch.optim.Adam(model.parameters(), lr=args['lr'])
  loss_fn = torch.nn.BCEWithLogitsLoss()

  best_model = None
  best_valid_acc = 0

  for epoch in range(1, 1 + args["epochs"]):
    print('Training...')
    loss = train(model, device, train_loader, optimizer, loss_fn)

    print('Evaluating...')
    train_result = eval(model, device, train_loader, evaluator)
    val_result = eval(model, device, valid_loader, evaluator)
    test_result = eval(model, device, test_loader, evaluator)

    train_acc, valid_acc, test_acc = train_result[dataset.eval_metric], val_result[dataset.eval_metric], test_result[dataset.eval_metric]
    if valid_acc > best_valid_acc:
        best_valid_acc = valid_acc
        best_model = copy.deepcopy(model)
    print(f'Epoch: {epoch:02d}, '
          f'Loss: {loss:.4f}, '
          f'Train: {100 * train_acc:.2f}%, '
          f'Valid: {100 * valid_acc:.2f}% '
          f'Test: {100 * test_acc:.2f}%')

Training...


Iteration: 100%|██████████| 1029/1029 [00:11<00:00, 92.60it/s]


Evaluating...


Iteration: 100%|██████████| 129/129 [00:00<00:00, 142.87it/s]


Epoch: 01, Loss: 0.0350, Train: 70.54%, Valid: 70.29% Test: 70.59%
Training...


...

Epoch: 29, Loss: 0.7072, Train: 82.92%, Valid: 78.25% Test: 75.07%
Training...


Iteration: 100%|██████████| 1029/1029 [00:43<00:00, 23.74it/s]


Evaluating...


Iteration: 100%|██████████| 129/129 [00:03<00:00, 38.45it/s]

Epoch: 30, Loss: 0.0252, Train: 83.66%, Valid: 77.98% Test: 74.30%


### Quesion 6： 你的最佳模型的验证/测试 ROC-AUC 分数多少？

运行下方的代码单元格，以查看你最优模型的预测结果，  
并将预测分别保存为两个文件：`ogbg-molhiv_graph_valid.csv` 和 `ogbg-molhiv_graph_test.csv`。

In [37]:
if 'IS_GRADESCOPE_ENV' not in os.environ:
  train_auroc = eval(best_model, device, train_loader, evaluator)[dataset.eval_metric]
  valid_auroc = eval(best_model, device, valid_loader, evaluator, save_model_results=True, save_file="valid")[dataset.eval_metric]
  test_auroc  = eval(best_model, device, test_loader, evaluator, save_model_results=True, save_file="test")[dataset.eval_metric]

  print(f'Best model: '
      f'Train: {100 * train_auroc:.2f}%, '
      f'Valid: {100 * valid_auroc:.2f}% '
      f'Test: {100 * test_auroc:.2f}%')

Iteration: 100%|██████████| 129/129 [00:00<00:00, 184.80it/s]


Saving Model Predictions


Iteration: 100%|██████████| 129/129 [00:00<00:00, 179.38it/s]

Saving Model Predictions
Best model: Train: 80.75%, Valid: 79.97% Test: 72.34%


### Question 7（选做）：在PyG中测试另外两种 global pooling

In [38]:
from torch_geometric.nn import global_max_pool, global_add_pool

# 测试 global_max_pool
class GCN_Graph_MaxPool(torch.nn.Module):
    def __init__(self, hidden_dim, output_dim, num_layers, dropout):
        super(GCN_Graph_MaxPool, self).__init__()

        # 加载分子图中原子的编码器
        self.node_encoder = AtomEncoder(hidden_dim)

        # 节点嵌入模型
        self.gnn_node = GCN(hidden_dim, hidden_dim,
            hidden_dim, num_layers, dropout, return_embeds=True)

        # 使用最大池化
        self.pool = global_max_pool

        # 输出层
        self.linear = torch.nn.Linear(hidden_dim, output_dim)

    def reset_parameters(self):
        self.gnn_node.reset_parameters()
        self.linear.reset_parameters()

    def forward(self, batched_data):
        x, edge_index, batch = batched_data.x, batched_data.edge_index, batched_data.batch
        
        x = self.node_encoder(x)
        node_embeddings = self.gnn_node(x, edge_index)
        graph_embeddings = self.pool(node_embeddings, batch)
        pred = self.linear(graph_embeddings)
        
        return pred

# 测试 global_add_pool
class GCN_Graph_AddPool(torch.nn.Module):
    def __init__(self, hidden_dim, output_dim, num_layers, dropout):
        super(GCN_Graph_AddPool, self).__init__()

        # 加载分子图中原子的编码器
        self.node_encoder = AtomEncoder(hidden_dim)

        # 节点嵌入模型
        self.gnn_node = GCN(hidden_dim, hidden_dim,
            hidden_dim, num_layers, dropout, return_embeds=True)

        # 使用求和池化
        self.pool = global_add_pool

        # 输出层
        self.linear = torch.nn.Linear(hidden_dim, output_dim)

    def reset_parameters(self):
        self.gnn_node.reset_parameters()
        self.linear.reset_parameters()

    def forward(self, batched_data):
        x, edge_index, batch = batched_data.x, batched_data.edge_index, batched_data.batch
        
        x = self.node_encoder(x)
        node_embeddings = self.gnn_node(x, edge_index)
        graph_embeddings = self.pool(node_embeddings, batch)
        pred = self.linear(graph_embeddings)
        
        return pred

# 测试最大池化
if 'IS_GRADESCOPE_ENV' not in os.environ:
    print("Testing global_max_pool...")
    model_max = GCN_Graph_MaxPool(args['hidden_dim'],
                dataset.num_tasks, args['num_layers'],
                args['dropout']).to(device)
    
    model_max.reset_parameters()
    optimizer = torch.optim.Adam(model_max.parameters(), lr=args['lr'])
    
    # 仅训练几个epoch进行测试
    for epoch in range(1, 6):
        print(f'Epoch: {epoch}')
        print('Training...')
        loss = train(model_max, device, train_loader, optimizer, loss_fn)
        
        print('Evaluating...')
        train_result = eval(model_max, device, train_loader, evaluator)
        val_result = eval(model_max, device, valid_loader, evaluator)
        test_result = eval(model_max, device, test_loader, evaluator)
        
        train_acc = train_result[dataset.eval_metric]
        valid_acc = val_result[dataset.eval_metric]
        test_acc = test_result[dataset.eval_metric]
        
        print(f'Loss: {loss:.4f}, Train: {100 * train_acc:.2f}%, Valid: {100 * valid_acc:.2f}% Test: {100 * test_acc:.2f}%')

# 测试求和池化
if 'IS_GRADESCOPE_ENV' not in os.environ:
    print("\nTesting global_add_pool...")
    model_add = GCN_Graph_AddPool(args['hidden_dim'],
                dataset.num_tasks, args['num_layers'],
                args['dropout']).to(device)
    
    model_add.reset_parameters()
    optimizer = torch.optim.Adam(model_add.parameters(), lr=args['lr'])
    
    # 仅训练几个epoch进行测试
    for epoch in range(1, 6):
        print(f'Epoch: {epoch}')
        print('Training...')
        loss = train(model_add, device, train_loader, optimizer, loss_fn)
        
        print('Evaluating...')
        train_result = eval(model_add, device, train_loader, evaluator)
        val_result = eval(model_add, device, valid_loader, evaluator)
        test_result = eval(model_add, device, test_loader, evaluator)
        
        train_acc = train_result[dataset.eval_metric]
        valid_acc = val_result[dataset.eval_metric]
        test_acc = test_result[dataset.eval_metric]
        
        print(f'Loss: {loss:.4f}, Train: {100 * train_acc:.2f}%, Valid: {100 * valid_acc:.2f}% Test: {100 * test_acc:.2f}%')

Testing global_max_pool...
Epoch: 1
Training...


Iteration: 100%|██████████| 1029/1029 [00:12<00:00, 80.20it/s] 


Evaluating...


Iteration: 100%|██████████| 129/129 [00:04<00:00, 29.89it/s]


Loss: 0.0633, Train: 75.02%, Valid: 71.33% Test: 71.03%
Epoch: 2
Training...


...

Loss: 0.2965, Train: 72.32%, Valid: 72.53% Test: 71.79%
Epoch: 5
Training...


Iteration: 100%|██████████| 1029/1029 [00:08<00:00, 125.59it/s]


Evaluating...


Iteration: 100%|██████████| 129/129 [00:00<00:00, 177.87it/s]

Loss: 0.0258, Train: 71.26%, Valid: 66.01% Test: 63.55%
